In [7]:
# ====================================================================
# SCRIPT : RECHERCHE DE FEATURES "AU FIL DE L'EAU"
# Objectif : Voir ce qui se passe en temps réel
# ====================================================================

import pandas as pd
import numpy as np
from tqdm import tqdm  # Pour la barre de progression
import warnings

warnings.filterwarnings('ignore') # On coupe les warnings de division par zéro

# 1. CHARGEMENT & ÉCHANTILLON
print("⏳ Chargement rapide...")
train_df = pd.read_csv('conversion_data_train.csv')
# On prend 10% pour que ça aille vite (suffisant pour trouver des corrélations)
df = train_df.sample(frac=0.50, random_state=42).copy()
y = df['converted']
df = df.drop('converted', axis=1)

# Encodage basique pour les calculs
df['country'] = df['country'].astype('category').cat.codes
df['source'] = df['source'].astype('category').cat.codes

print(f"🔬 Analyse sur {len(df)} lignes. C'est parti !\n")

# Colonnes numériques
cols = ['age', 'total_pages_visited']
best_corr = 0
best_feature = ""

# Liste pour stocker les découvertes
discoveries = []

# 2. BOUCLE DE RECHERCHE (Avec barre de progression)
# On va tester des opérations simples et doubles

# A. Transformations Simples (Unary)
ops_unary = ['log', 'sqrt', 'square', 'inverse']
print("--- PHASE 1 : Transformations Simples ---")

for col in cols:
    for op in ops_unary:
        name = f"{op}({col})"
        
        if op == 'log':
            feat = np.log(df[col] + 1) # +1 pour éviter log(0)
        elif op == 'sqrt':
            feat = np.sqrt(df[col])
        elif op == 'square':
            feat = df[col] ** 2
        elif op == 'inverse':
            feat = 1 / (df[col] + 0.1)
            
        # Calcul de corrélation (Absolue)
        corr = abs(feat.corr(y))
        
        if corr > 0.1: # Si c'est pertinent
            print(f"   Test {name:<20} | Corrélation : {corr:.4f}")
            discoveries.append((name, corr))

# B. Interactions (Binary)
print("\n--- PHASE 2 : Interactions (Croisements) ---")
# On ajoute les colonnes catégorielles encodées pour les interactions
all_cols = cols + ['country', 'source']
import itertools

# On crée toutes les paires possibles
combinations = list(itertools.combinations(all_cols, 2))

# Barre de chargement visuelle
for col1, col2 in tqdm(combinations, desc="Test des interactions"):
    # Multiplication
    feat_mul = df[col1] * df[col2]
    corr_mul = abs(feat_mul.corr(y))
    
    # Division
    feat_div = df[col1] / (df[col2] + 0.1)
    corr_div = abs(feat_div.corr(y))
    
    # Affichage en direct si c'est fort
    if corr_mul > 0.35: # Seuil d'intérêt
        print(f"   🚀 PÉPITE : {col1} * {col2} -> Corr: {corr_mul:.4f}")
        discoveries.append((f"{col1} * {col2}", corr_mul))
        
    if corr_div > 0.35:
        print(f"   🚀 PÉPITE : {col1} / {col2} -> Corr: {corr_div:.4f}")
        discoveries.append((f"{col1} / {col2}", corr_div))

# 3. BILAN
print("\n🏆 TOP 5 DES MEILLEURES FORMULES TROUVÉES :")
discoveries.sort(key=lambda x: x[1], reverse=True)
for name, score in discoveries[:5]:
    print(f"   ⭐ {name} : {score:.4f}")

⏳ Chargement rapide...
🔬 Analyse sur 142290 lignes. C'est parti !

--- PHASE 1 : Transformations Simples ---
   Test log(total_pages_visited) | Corrélation : 0.3627
   Test sqrt(total_pages_visited) | Corrélation : 0.4302
   Test square(total_pages_visited) | Corrélation : 0.6742
   Test inverse(total_pages_visited) | Corrélation : 0.1828

--- PHASE 2 : Interactions (Croisements) ---


Test des interactions: 100%|██████████| 6/6 [00:00<00:00, 474.42it/s]

   🚀 PÉPITE : age * total_pages_visited -> Corr: 0.4092
   🚀 PÉPITE : total_pages_visited * country -> Corr: 0.4859

🏆 TOP 5 DES MEILLEURES FORMULES TROUVÉES :
   ⭐ square(total_pages_visited) : 0.6742
   ⭐ total_pages_visited * country : 0.4859
   ⭐ sqrt(total_pages_visited) : 0.4302
   ⭐ age * total_pages_visited : 0.4092
   ⭐ log(total_pages_visited) : 0.3627
